# BigQuery com dados públicos

`Atenção: Se você estiver trabalhar localmente e usando uma conta sandbox do BigQuery, copie e cole as consultas diretamente no BigQuery Studio.`


## Pré-requisitos

Antes de executar, instale as dependências com uv:

`uv sync`

E depois abra o notebook com:

`uv run jupyter lab`

## Configuração inicial

Se a autenticação não estiver configurada, rode:

gcloud auth application-default login


In [ ]:

from google.cloud import bigquery

client = bigquery.Client() 
print("Cliente do BigQuery pronto")

## Exemplo 1: consulta simples

In [ ]:
query = """
SELECT
  name,
  year,
  gender,
  number
FROM `bigquery-public-data.usa_names.usa_1910_current`
WHERE year = 2020
ORDER BY number DESC
LIMIT 10
"""

df = client.query(query).to_dataframe()
df.head()

## Exemplo 2: agregação

In [ ]:
query = """
SELECT
  year,
  gender,
  SUM(number) AS total_names
FROM `bigquery-public-data.usa_names.usa_1910_current`
WHERE year BETWEEN 2010 AND 2020
GROUP BY year, gender
ORDER BY year, gender
"""

agg_df = client.query(query).to_dataframe()
agg_df.head()

## Exemplo 3: window functions

In [ ]:
query = """
SELECT
  year,
  name,
  number,
  RANK() OVER (PARTITION BY year ORDER BY number DESC) AS rank_in_year
FROM `bigquery-public-data.usa_names.usa_1910_current`
WHERE year = 2020
ORDER BY year, rank_in_year
LIMIT 10
"""

window_df = client.query(query).to_dataframe()
window_df.head()

## Exemplo 4: pipe syntax

In [ ]:
query = """
FROM `bigquery-public-data.usa_names.usa_1910_current`
|> WHERE year = 2020
|> ORDER BY number DESC
|> LIMIT 10
"""

window_df = client.query(query).to_dataframe()
window_df.head()

## Exemplo 5: clusterização e particionamento

Consulte os detalhes da tabela para entender como ela é particionada!

In [ ]:
query = """
SELECT title, SUM(views) AS views 
FROM `bigquery-public-data.wikipedia.pageviews_2026` 
WHERE datehour >= TIMESTAMP("2026-07-31 00:00:00")
  AND wiki = 'br' 
GROUP BY title 
ORDER BY views DESC 
LIMIT 100;
"""

partition_example_df = client.query(query).to_dataframe()
partition_example_df

## Boas práticas

Algumas boas práticas ajudam a manter consultas mais baratas e legíveis.

In [ ]:
# Boas práticas:
# 1. Filtre cedo para reduzir bytes lidos.
# 2. Selecione só as colunas necessárias.
# 3. Prefira consultas legíveis e explícitas.
# 4. Observe o plano de execução quando a performance importar.